# Global-norm clipping with a non-finite guard

Clip a gradient pytree by its **global** L2 norm, the norm of all leaves concatenated into one vector, and protect the step against overflow: if any gradient is `inf` or `nan`, the step should be skipped.

Write `clip_global(grads, max_norm)`:

- `grads`: a pytree (nested dicts) of float arrays of any shape.
- Global norm: $n = \sqrt{\sum_{\text{leaves}} \sum_i g_i^2}$.
- If $n$ is finite: scale every leaf by $\min(1,\ C / n)$, where $C$ = `max_norm` $> 0$. When $n = 0$ the factor is 1.
- If $n$ is not finite (`inf` or `nan`): return all-zero gradients.

Return `(clipped, norm, finite)`:

- `clipped`: same structure as `grads`, float32.
- `norm`: 0-d float32 array, the global norm **before** clipping (it may be `inf` or `nan`).
- `finite`: 0-d bool array.

The function must work under `jax.jit`, so no Python `if` on array values: use `jnp.where`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def clip_global(grads, max_norm):
    """Return (clipped pytree, global norm, finite flag)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["a", {"__nd__": [3.0], "dtype": "float64"}], ["b", {"__nd__": [4.0], "dtype": "float64"}]]}, 1.0], "e": {"__tup__": [{"__dict__": [["a", {"__nd__": [0.6000000238418579], "dtype": "float32"}], ["b", {"__nd__": [0.800000011920929], "dtype": "float32"}]]}, {"__nd__": 5.0, "dtype": "float32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["a", {"__nd__": [0.3], "dtype": "float64"}], ["b", {"__nd__": [0.4], "dtype": "float64"}]]}, 1.0], "e": {"__tup__": [{"__dict__": [["a", {"__nd__": [0.30000001192092896], "dtype": "float32"}], ["b", {"__nd__": [0.4000000059604645], "dtype": "float32"}]]}, {"__nd__": 0.5, "dtype": "float32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["enc", {"__dict__": [["w", {"__nd__": [[0.39257939815601534, -2.4354174163439515, -0.11022594532382894], [0.16291581767584848, 0.5612247602630475, -0.816089756439701], [-1.3280342642624956, -0.9507340805091575, 0.5480409641909273], [-0.8518355089919037, -0.7641438115235788, 1.1437655777004572]], "dtype": "float64"}], ["b", {"__nd__": [-0.7861575894791125, -0.8986326207536771, -0.09123756351337108], "dtype": "float64"}]]}], ["head", {"__nd__": [0.19861051640677327, 1.0376296586712488, -0.8793276811910162, -1.5681545944352957, -0.12249665156557239], "dtype": "float64"}]]}, 1.0], "e": {"__tup__": [{"__dict__": [["enc", {"__dict__": [["b", {"__nd__": [-0.18283693492412567, -0.2089952975511551, -0.021219149231910706], "dtype": "float32"}], ["w", {"__nd__": [[0.09130232781171799, -0.5664058327674866, -0.025635285302996635], [0.03788938373327255, 0.13052423298358917, -0.1897982656955719], [-0.3088613748550415, -0.22111254930496216, 0.12745808064937592], [-0.1981116682291031, -0.1777171790599823, 0.2660059332847595]], "dtype": "float32"}]]}], ["head", {"__nd__": [0.04619091376662254, 0.241321861743927, -0.2045055329799652, -0.36470624804496765, -0.028489086776971817], "dtype": "float32"}]]}, {"__nd__": 4.299774646759033, "dtype": "float32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["enc", {"__dict__": [["w", {"__nd__": [[6.436368157294202, -14.81526543249413, -4.872630157846969], [3.0619578810330355, 2.310243000221297, 0.8335899091739197], [-2.568895173775524, -7.993464580715008, -9.281380009611976], [11.562622342720783, -11.172097148854832, 12.280000985175473]], "dtype": "float64"}], ["b", {"__nd__": [-13.150584358653276, 29.03581786972847, -17.38110928883392], "dtype": "float64"}]]}], ["head", {"__nd__": [16.04274299454078, -8.970721577902044, -1.1561548050669124, -0.6555257091474957, -6.522226650767547], "dtype": "float64"}]]}, 0.5], "e": {"__tup__": [{"__dict__": [["enc", {"__dict__": [["b", {"__nd__": [-0.12979638576507568, 0.2865837812423706, -0.17155170440673828], "dtype": "float32"}], ["w", {"__nd__": [[0.06352701038122177, -0.14622679352760315, -0.04809290170669556], [0.030221549794077873, 0.022802116349339485, 0.008227539248764515], [-0.025355016812682152, -0.07889556139707565, -0.09160729497671127], [0.11412318050861359, -0.11026869714260101, 0.12120371311903]], "dtype": "float32"}]]}], ["head", {"__nd__": [0.15834201872348785, -0.08854109793901443, -0.011411257088184357, -0.006470044143497944, -0.06437443196773529], "dtype": "float32"}]]}, {"__nd__": 50.65851593017578, "dtype": "float32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["a", {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}], ["b", {"__dict__": [["c", {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}]]}]]}, 1.0], "e": {"__tup__": [{"__dict__": [["a", {"__nd__": [0.0, 0.0, 0.0], "dtype": "float32"}], ["b", {"__dict__": [["c", {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float32"}]]}]]}, {"__nd__": 0.0, "dtype": "float32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["enc", {"__dict__": [["w", {"__nd__": [[1.101262453505847, 0.3384312766461778, -0.5399715152535035], [-1.2602418568524327, -1.8946212698392553, 0.018638290983285614], [-0.8105670995116028, -0.8721559599345132, -0.22196950708389104], [-0.05184602813201771, -2.2767828157758307, 0.9251465905764266]], "dtype": "float64"}], ["b", {"__nd__": [-2.026845605910026, {"__f__": "inf"}, 0.5905681115998624], "dtype": "float64"}]]}], ["head", {"__nd__": [-0.47184451813331413, 1.342996537380774, 0.020786461917443023, 0.6909209676232841, 0.10722707944720507], "dtype": "float64"}]]}, 1.0], "e": {"__tup__": [{"__dict__": [["enc", {"__dict__": [["b", {"__nd__": [0.0, 0.0, 0.0], "dtype": "float32"}], ["w", {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float32"}]]}], ["head", {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float32"}]]}, {"__nd__": {"__f__": "inf"}, "dtype": "float32"}, {"__nd__": false, "dtype": "bool"}]}}, {"a": [{"__dict__": [["enc", {"__dict__": [["w", {"__nd__": [[1.096531380130275, 1.0614224543758295, -0.9073341413930657], [-0.6124917784436158, 0.3432270521124315, -0.21001015163517062], [-2.284914216554451, 2.0261175349217764, -2.174977373868806], [-2.081542853744764, -1.2761124197329383, 0.5648512404463458]], "dtype": "float64"}], ["b", {"__nd__": [1.7960125066302708, -0.2354150525900867, -0.3910773505445041], "dtype": "float64"}]]}], ["head", {"__nd__": [{"__f__": "nan"}, -0.3668597432540227, 0.08236522460221347, -1.1399714593630075, 0.438257432619766], "dtype": "float64"}]]}, 5.0], "e": {"__tup__": [{"__dict__": [["enc", {"__dict__": [["b", {"__nd__": [0.0, 0.0, 0.0], "dtype": "float32"}], ["w", {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float32"}]]}], ["head", {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float32"}]]}, {"__nd__": {"__f__": "nan"}, "dtype": "float32"}, {"__nd__": false, "dtype": "bool"}]}}, {"a": [{"__dict__": [["s", {"__nd__": -6.0, "dtype": "float64"}], ["v", {"__nd__": [8.0, 0.0], "dtype": "float64"}]]}, 5.0], "e": {"__tup__": [{"__dict__": [["s", {"__nd__": -3.0, "dtype": "float32"}], ["v", {"__nd__": [4.0, 0.0], "dtype": "float32"}]]}, {"__nd__": 10.0, "dtype": "float32"}, {"__nd__": true, "dtype": "bool"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(clip_global, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")